# Protocolo de Firmas Digitales usando Curvas Elípticas (ECDSA)

En este notebook, mostraremos el proceso para generar las llaves pública y privada, así como el proceso de generar una firma digital y su comprobación correspondiente usando el protocolo ECDSA. Empezaremos por cargar las funciones que necesitamos:

In [89]:
import sys
import importlib.util

if 'google.colab' in sys.modules:
    runtime = "Google Colab"
    if importlib.util.find_spec("cryptocalc") is None:
        print("   Installing MA2006B from GitHub\n")
        !pip install git+https://github.com/Krul-dev/MA2006B.git
else:
    runtime = "Local environment"

import cryptocalc

print(f"\n========= Notebook execution context =========")
print(f"              Runtime: {runtime}")
print(f"       Python version: {sys.version.split()[0]}")
print(f"   CryptoCalc version: {cryptocalc.__version__}")

from cryptocalc import (
    EllipticCurve,
    SECP256K1,
    generate_elliptic_private_key,
    sha256_of_sentence,
    gcd,
    divide_mod,
)


========= Notebook execution context =========
              Runtime: Local environment
       Python version: 3.14.3
   CryptoCalc version: 0.1.0


Para este ejemplo, nuestra curva elíptica $E$ corresponderá a la curva `secp256k1` la cual se encuentra especificada en el documento [SEC 2: Recommended Elliptic Curve Domain Parameters.](https://www.secg.org/sec2-v2.pdf)

In [90]:
p = SECP256K1["p"]
c1 = SECP256K1["a"]
c0 = SECP256K1["b"]
G = SECP256K1["G"]
n = SECP256K1["n"]
h = SECP256K1["h"]
curve = EllipticCurve(p,c1,c0)

print(f"Primo Base (p): {p}\n")
print(f"Coeficiente del término lineal: {c1}\n")
print(f"Coeficiente del término constante: {c0}\n")
print(f"Generador (G): {G}\n")
print(f"Orden del Generador (n): {n}\n")


Primo Base (p): 115792089237316195423570985008687907853269984665640564039457584007908834671663

Coeficiente del término lineal: 0

Coeficiente del término constante: 7

Generador (G): (55066263022277343669578718895168534326250603453777594175500187360389116729240, 32670510020758816978083085130507043184471273380659243275938904335757337482424)

Orden del Generador (n): 115792089237316195423570985008687907852837564279074904382605163141518161494337



Generamos ahora la llave privada de ALicia de manera aleatoria.

In [91]:
d = generate_elliptic_private_key(n)

print(f"Llave privada de Alicia (d): {d}\n")

Llave privada de Alicia (d): 86354593674780183959191151169357724926327298041792611422534559802781230424884



La llave pública de Alicia estará dada por $Q =d G \in E(\mathbb{F}_{p})$

In [92]:
Q = curve.multiplication(d,G)

print(f"Llave pública de Alicia (Q): {Q}\n")

Llave pública de Alicia (Q): (88667144575910615510267273275012225456584840114924792315518082286766745627839, 84511052372714249511819808436518050124248993715479796993514581823512853569337)



Para ilustrar el algoritmo de generación de firmas digitales, supongamos que Alicia desea firmar el siguiente mensaje llano $m$:

In [93]:
m = "Hello World!"
h = sha256_of_sentence(m)

print(f"Mensaje llano (m): {m}\n")
print(f"Hash del mensaje llano (h): {h}\n")

Mensaje llano (m): Hello World!

Hash del mensaje llano (h): 57676413081093003148005107550719583540116985236696423860923466490497932824681



Ahora generamos un número aleatorio $1 < k < n$ tal que $\operatorname{mcd}(n,k) = 1$.

In [94]:
k = generate_elliptic_private_key(n)
while(gcd(n,k) != 1):
    k = generate_elliptic_private_key(n)

A continuación, calculamos el punto sobre la curva eliptica $R = kG$ y definimos $r = R_{1}$ , es decir, la primera coordenada del punto $R$

In [95]:
R = curve.multiplication(k, G)
r = R[0]

Calculamos ahora el valor
$$
s = \frac{h + dr}{k} \mod n
$$
Si $\operatorname{mcd}(n,s) \neq 1$, Alicia debe elegir un nuevo valor de $k$ y repetir este proceso.

In [96]:
s = divide_mod((h + d*r, n), (k, n))[0]

print(f"El máximo común divisor de n y s es:", gcd(n,s))

El máximo común divisor de n y s es: 1


Con esta información, podemos ahora generar el *mensaje firmado*
$$
\text{mensaje\_firmado} = (m,r,s)
$$

In [97]:
mensaje_firmado = (m, r, s)

print(f"El mensaje firmado es: {mensaje_firmado}\n")

El mensaje firmado es: ('Hello World!', 16153416039858850470317524994535254404457224182174532376807101922330982920380, 51987456478254901800755336742109639141448094840599089552070955927657919665049)



Procederemos ahora a verificar la firma de este mensaje. Para empezar, Beto debe calcular el hash del mensaje firmado.

In [98]:
h = sha256_of_sentence(mensaje_firmado[0])

print(f"Hash del mensaje firmado (h): {h}\n")

Hash del mensaje firmado (h): 57676413081093003148005107550719583540116985236696423860923466490497932824681



Notamos que este es el mismo hash que el obtenido anteriormente. Calculamos ahora las constantes 
$$
\begin{aligned}
w & =  s^{-1} \mod n \\
u_{1} & =  hw \mod n \\
u_{2} & =  rw \mod n \\
P & =  u_{1}G + u_{2}Q \in E(\mathbb{F}_{p})
\end{aligned}
$$
Si $P = \infty,$ entonces la firma es automáticamente inválida.

In [99]:
r = mensaje_firmado[1]
s = mensaje_firmado[2]

w = divide_mod((1, n), (s, n))[0]
u1 = (h*w) % n
u2 = (r*w) % n
P = curve.addition(curve.multiplication(u1, G), curve.multiplication(u2, Q))

print(f"El valor del punto P es: {P}")

El valor del punto P es: (16153416039858850470317524994535254404457224182174532376807101922330982920380, 88031037333459423176119342000851561456498954825493092794363006303573122333203)


Definimos ahora $p = P_{1}$, es decir, la primera coordenada del punto $P$. Si $p = r$, entonces la firma es válida. En caso contrario, la firma es inválida.

In [100]:
p = P[0]

if p == r:
    verificacion_firma = "la firma es válida"
else:
    verificacion_firma = "la firma es inválida"

print(f"El valor de r es: {r}\n")
print(f"El valor de p es: {p}\n")
print("Por lo tanto,", verificacion_firma)

El valor de r es: 16153416039858850470317524994535254404457224182174532376807101922330982920380

El valor de p es: 16153416039858850470317524994535254404457224182174532376807101922330982920380

Por lo tanto, la firma es válida
